# 08 — Evaluation and Experimental Comparison

This notebook evaluates the three proposed strategies on the held-out test set. It reports answerability metrics, false-answer and false-abstention rates, QA Exact Match and token F1, confusion matrices, precision-recall and risk-coverage plots.

## 1. Evaluation objectives

The proposal asks for:
- answerability precision, recall and F1;
- answer exact match and token-level F1 on answerable questions;
- false-answer rate on unanswerable questions;
- false-abstention rate on answerable questions;
- overall risk-coverage results;
- confusion matrix, precision-recall and risk-coverage visualizations.

We evaluate all strategies on the same held-out test questions and fixed QA outputs.

In [ ]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    precision_recall_fscore_support, confusion_matrix,
    precision_recall_curve, average_precision_score
)

ROOT=Path.cwd()
if ROOT.name=="notebooks": ROOT=ROOT.parent
SPLITS=ROOT/"data"/"splits"
RESULTS=ROOT/"results"

test=pd.read_csv(SPLITS/"test.csv")
test["answers"]=test["answers"].apply(json.loads)
test["is_answerable"]=test["is_answerable"].astype(bool)

ret=pd.read_csv(RESULTS/"test_retrieval_features.csv")
# Drop columns already in test to avoid _x/_y duplicates
overlap=[c for c in ret.columns if c in test.columns and c!="question_id"]
ret=ret.drop(columns=overlap)

qa=pd.read_csv(RESULTS/"test_qa_predictions.csv")
test=test.merge(ret,on="question_id",validate="one_to_one").merge(
    qa,on="question_id",validate="one_to_one"
)
threshold_decisions=pd.read_csv(RESULTS/"test_threshold_decisions.csv")
classifier_pred=pd.read_csv(RESULTS/"test_classifier_predictions.csv")
test=test.merge(threshold_decisions,on="question_id",validate="one_to_one")
test=test.merge(classifier_pred,on="question_id",validate="one_to_one")
print("Test examples:",len(test))

## 2. Define strategy decisions

`True` means the system is allowed to answer; `False` means it abstains. The baseline always attempts an answer. The threshold and classifier decisions were selected using validation data in previous notebooks.

In [ ]:
strategies={
    "Always answer": np.ones(len(test),dtype=bool),
    "BM25 threshold": test["bm25_threshold_answer"].astype(bool).to_numpy(),
    "Answerability classifier": test["classifier_answer"].astype(bool).to_numpy(),
}
y_true=test.is_answerable.astype(bool).to_numpy()

def answerability_metrics(y_true,decisions):
    p,r,f,_=precision_recall_fscore_support(
        y_true,decisions,average="binary",zero_division=0
    )
    unans=~y_true
    ans=y_true
    false_answer_rate=(decisions[unans].mean() if unans.any() else np.nan)
    false_abstention_rate=((~decisions[ans]).mean() if ans.any() else np.nan)
    return {
        "precision":p,"recall":r,"f1":f,
        "coverage":decisions.mean(),
        "false_answer_rate":false_answer_rate,
        "false_abstention_rate":false_abstention_rate,
    }

summary=pd.DataFrame([
    {"strategy":name,**answerability_metrics(y_true,pred)}
    for name,pred in strategies.items()
])
display(summary)
summary.to_csv(RESULTS/"strategy_comparison.csv",index=False)

## 3. Exact Match and token F1

For answerable questions, compare the fixed QA prediction with all accepted reference answers. We use the standard SQuAD-style normalization: lowercase, remove punctuation/articles, and normalize whitespace. Token F1 measures overlap between predicted and reference tokens.

These are answer-quality metrics, separate from the answerability decision metrics.

In [ ]:
ARTICLES={"a","an","the"}
PUNCT=re.compile(r"[^\w\s]")

def normalize_answer(s):
    s=str(s).lower()
    s=PUNCT.sub(" ",s)
    tokens=[t for t in s.split() if t not in ARTICLES]
    return " ".join(tokens)

def exact_match(pred, gold):
    return int(normalize_answer(pred)==normalize_answer(gold))

def token_f1(pred,gold):
    p=normalize_answer(pred).split()
    g=normalize_answer(gold).split()
    if not p or not g:
        return float(p==g)
    common=Counter(p) & Counter(g)
    overlap=sum(common.values())
    if overlap==0: return 0.0
    precision=overlap/len(p)
    recall=overlap/len(g)
    return 2*precision*recall/(precision+recall)

from collections import Counter

def best_reference_score(pred, answers, metric):
    refs=[a.get("text","") for a in answers]
    if not refs: return np.nan
    return max(metric(pred,ref) for ref in refs)

answerable=test[test.is_answerable].copy()
answerable["exact_match"]=answerable.apply(
    lambda r:best_reference_score(r.predicted_answer,r.answers,exact_match),axis=1
)
answerable["token_f1"]=answerable.apply(
    lambda r:best_reference_score(r.predicted_answer,r.answers,token_f1),axis=1
)
print("QA Exact Match on answerable questions:",answerable.exact_match.mean())
print("QA token F1 on answerable questions:",answerable.token_f1.mean())

## 4. Strategy-specific selective answer quality

For each strategy, compute answer correctness among the questions it chooses to answer. Here, a QA answer is counted correct if token F1 is greater than zero; for a stricter analysis, also report exact match. A production-quality study may choose a different correctness criterion, but it must be stated explicitly.

In [ ]:
quality=test[["question_id","is_answerable","predicted_answer","answers"]].copy()
quality["exact_match"]=quality.apply(
    lambda r:best_reference_score(r.predicted_answer,r.answers,exact_match),axis=1
)
quality["token_f1"]=quality.apply(
    lambda r:best_reference_score(r.predicted_answer,r.answers,token_f1),axis=1
)
quality["answer_correct_token_overlap"]=quality.token_f1.fillna(0)>0

quality_rows=[]
for name,decision in strategies.items():
    answered=quality.loc[decision]
    quality_rows.append({
        "strategy":name,
        "answered_count":int(decision.sum()),
        "mean_exact_match_answered":answered.exact_match.mean(),
        "mean_token_f1_answered":answered.token_f1.mean(),
        "token_overlap_accuracy_answered":answered.answer_correct_token_overlap.mean()
    })
quality_summary=pd.DataFrame(quality_rows)
display(quality_summary)
quality_summary.to_csv(RESULTS/"answer_quality_by_strategy.csv",index=False)

## 5. Confusion matrices

Positive class means answerable. A false positive is an unanswerable example the system decides to answer; a false negative is an answerable example the system refuses.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,(name,pred) in zip(axes,strategies.items()):
    cm=confusion_matrix(y_true,pred,labels=[False,True])
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",cbar=False,ax=ax,
                xticklabels=["Abstain","Answer"],
                yticklabels=["Unanswerable","Answerable"])
    ax.set_title(name)
    ax.set_xlabel("System decision")
    ax.set_ylabel("Gold label")
plt.tight_layout()
plt.show()

## 6. Precision-recall curve for classifier

The curve uses the classifier's continuous answerability probability. It illustrates the trade-off across thresholds. The final classifier threshold was selected on validation, not from this test curve; use this plot descriptively and do not retune based on it.

In [ ]:
precision,recall,_=precision_recall_curve(
    y_true,test.answerability_probability
)
ap=average_precision_score(y_true,test.answerability_probability)
plt.figure(figsize=(7,5))
plt.plot(recall,precision,label=f"Average precision={ap:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Test precision-recall curve (descriptive)")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Risk-coverage curve

Coverage is the fraction of examples the system answers. Risk is the fraction of answered examples that are incorrect under our defined answer-correctness criterion. We order examples by answerability confidence and progressively allow more answers.

This curve is based on the classifier score and is descriptive on test data; the operating threshold remains the one selected on validation.

In [ ]:
order=np.argsort(-test.answerability_probability.to_numpy())
correct=quality.answer_correct_token_overlap.to_numpy()[order]
coverages=[]
risks=[]
for n in range(1,len(order)+1):
    coverages.append(n/len(order))
    risks.append(1-correct[:n].mean())

plt.figure(figsize=(7,5))
plt.plot(coverages,risks)
plt.xlabel("Coverage")
plt.ylabel("Risk among answered examples")
plt.title("Risk–coverage curve (test, classifier ranking)")
plt.tight_layout()
plt.show()

## 8. Save final evaluation artifacts

Save summary tables and a compact per-question evaluation file for manual error analysis.

In [ ]:
evaluation_rows=test[[
    "question_id","question","is_answerable","predicted_answer",
    "answerability_probability","classifier_answer",
    "bm25_threshold_answer","top_bm25_score","qa_score"
]].copy()
evaluation_rows.to_csv(RESULTS/"test_evaluation_rows.csv",index=False)
print("Saved evaluation outputs in",RESULTS)

## 9. Interpretation checklist

- Does the threshold strategy reduce false answers relative to always-answer?
- What is the cost in false abstentions?
- Does the classifier improve the precision/coverage balance?
- Are answers still factually correct when the system decides to answer?
- Which strategy gives the best risk-coverage behavior?
- What limitations arise from using the original SQuAD paired-passage label as a proxy?

**Next:** Notebook 09 performs manual error analysis and prepares examples for the report.